In [ ]:
# ============================================
# Segmentación, cálculos analíticos
# y fuentes para Tableau
# ============================================

import pandas as pd

df = pd.read_csv('../data/interim/train_clean.csv')

# Asegurar tipos correctos
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], errors='coerce')

print('Dataset cargado:', df.shape)

# --------------------------------------------
# 0. Limpieza de columna redundante
# --------------------------------------------
# 'Month Name' y 'Month_Name' son idénticas -> eliminamos una
if 'Month Name' in df.columns:
    df = df.drop(columns=['Month Name'])

print('Columnas finales:', df.columns.tolist())

Dataset cargado: (9800, 26)
Columnas finales: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Year', 'Month', 'Nivel_Ventas', 'Month_Name', 'Quarter', 'Region_Category', 'Shipping_Time']


In [3]:
# --------------------------------------------
# 1. Tabla resumen: Region x Category x Year x Quarter
# --------------------------------------------
resumen_rcq = (
    df.groupby(['Region', 'Category', 'Year', 'Quarter'], as_index=False)
      .agg(
          Total_Sales=('Sales', 'sum'),
          N_Transacciones=('Sales', 'count'),
          Ticket_Promedio=('Sales', 'mean')
      )
)

# % de participación de cada categoría dentro de su región/periodo
resumen_rcq['Pct_Participacion_Categoria'] = (
    resumen_rcq.groupby(['Region', 'Year', 'Quarter'])['Total_Sales']
               .transform(lambda x: x / x.sum() * 100)
)

display(resumen_rcq.head())

,Region,Category,Year,Quarter,Total_Sales,N_Transacciones,Ticket_Promedio,Pct_Participacion_Categoria
0,Central,Furniture,2015,2015Q1,4584.9580,15,305.663867,53.346081
1,Central,Furniture,2015,2015Q2,7995.1026,20,399.755130,46.670464
2,Central,Furniture,2015,2015Q3,9340.2058,22,424.554809,21.219242
3,Central,Furniture,2015,2015Q4,10894.1012,34,320.414741,32.836121
4,Central,Furniture,2016,2016Q1,5867.0220,16,366.688875,51.831948


In [4]:
# --------------------------------------------
# 2. Ventas acumuladas (running total) por año/quarter
# --------------------------------------------
resumen_anual = (
    df.groupby(['Year', 'Quarter'], as_index=False)['Sales']
      .sum()
      .rename(columns={'Sales': 'Total_Sales'})
)
resumen_anual = resumen_anual.sort_values(['Year', 'Quarter'])
resumen_anual['Ventas_Acumuladas'] = (
    resumen_anual.groupby('Year')['Total_Sales'].cumsum()
)

display(resumen_anual)

,Year,Quarter,Total_Sales,Ventas_Acumuladas
0,2015,2015Q1,73931.3960,73931.3960
1,2015,2015Q2,85874.0936,159805.4896
2,2015,2015Q3,142522.6063,302328.0959
3,2015,2015Q4,177528.1122,479856.2081
4,2016,2016Q1,62357.6870,62357.6870
5,2016,2016Q2,87713.3730,150071.0600
6,2016,2016Q3,128560.2072,278631.2672
7,2016,2016Q4,180804.7382,459436.0054
8,2017,2017Q1,92686.3650,92686.3650
9,2017,2017Q2,135061.1610,227747.5260


In [5]:
# --------------------------------------------
# 3. Variación % interanual (YoY) por categoría
# --------------------------------------------
ventas_anuales_cat = (
    df.groupby(['Category', 'Year'], as_index=False)['Sales']
      .sum()
      .rename(columns={'Sales': 'Total_Sales'})
      .sort_values(['Category', 'Year'])
)
ventas_anuales_cat['Sales_Prev_Year'] = (
    ventas_anuales_cat.groupby('Category')['Total_Sales'].shift(1)
)
ventas_anuales_cat['YoY_Growth_Pct'] = (
    (ventas_anuales_cat['Total_Sales'] - ventas_anuales_cat['Sales_Prev_Year'])
    / ventas_anuales_cat['Sales_Prev_Year'] * 100
)

display(ventas_anuales_cat)

,Category,Year,Total_Sales,Sales_Prev_Year,YoY_Growth_Pct
0,Furniture,2015,156477.8811,NaN,NaN
1,Furniture,2016,164053.8674,156477.8811,4.841570
2,Furniture,2017,195813.0400,164053.8674,19.358990
3,Furniture,2018,212313.7872,195813.0400,8.426787
4,Office Supplies,2015,149512.8200,NaN,NaN
5,Office Supplies,2016,133124.4070,149512.8200,-10.961209
6,Office Supplies,2017,182417.5660,133124.4070,37.027890
7,Office Supplies,2018,240367.5410,182417.5660,31.767760
8,Technology,2015,173865.5070,NaN,NaN
9,Technology,2016,162257.7310,173865.5070,-6.676296


In [6]:
# --------------------------------------------
# 4. Ranking de regiones por ventas totales
# --------------------------------------------
ranking_regiones = (
    df.groupby('Region', as_index=False)['Sales']
      .sum()
      .rename(columns={'Sales': 'Total_Sales'})
      .sort_values('Total_Sales', ascending=False)
      .reset_index(drop=True)
)
ranking_regiones['Ranking'] = ranking_regiones.index + 1

display(ranking_regiones)

,Region,Total_Sales,Ranking
0,West,710219.6845,1
1,East,669518.7260,2
2,Central,492646.9132,3
3,South,389151.4590,4


In [7]:
# --------------------------------------------
# 5. Estacionalidad pura (Quarter sin año) por categoría
# --------------------------------------------
df['Quarter_Label'] = df['Quarter'].astype(str).str.extract(r'(Q\d)')

estacionalidad = (
    df.groupby(['Quarter_Label', 'Category'], as_index=False)['Sales']
      .mean()
      .rename(columns={'Sales': 'Avg_Sales'})
)

display(estacionalidad)

,Quarter_Label,Category,Avg_Sales
0,Q1,Furniture,338.102160
1,Q1,Office Supplies,134.618247
2,Q1,Technology,561.045362
3,Q2,Furniture,327.841787
4,Q2,Office Supplies,107.951457
5,Q2,Technology,431.838351
6,Q3,Furniture,355.055487
7,Q3,Office Supplies,120.654693
8,Q3,Technology,412.877965
9,Q4,Furniture,363.636799


In [ ]:
# --------------------------------------------
# 6. Validaciones de integridad 
# --------------------------------------------

total_original = df['Sales'].sum()
print(f"Suma total Sales (dataset original): {total_original:,.2f}\n")

# --- Validación 1: resumen_rcq ---
print("=== resumen_region_category_quarter ===")
print("Filas:", len(resumen_rcq))
print("Combinaciones únicas esperadas:", df.groupby(['Region','Category','Year','Quarter']).ngroups)
print(f"Suma Total_Sales: {resumen_rcq['Total_Sales'].sum():,.2f}")
print("¿Coincide con original?", abs(resumen_rcq['Total_Sales'].sum() - total_original) < 0.01)
print()

# --- Validación 2: ventas_acumuladas_anual ---
print("=== ventas_acumuladas_anual ===")
print("Filas:", len(resumen_anual))
print("Combinaciones únicas esperadas:", df.groupby(['Year','Quarter']).ngroups)
print(f"Suma Total_Sales (sin acumular): {resumen_anual['Total_Sales'].sum():,.2f}")
print("¿Coincide con original?", abs(resumen_anual['Total_Sales'].sum() - total_original) < 0.01)
# Verificar que el último acumulado de cada año == suma del año
ultimo_acumulado_por_año = resumen_anual.groupby('Year')['Ventas_Acumuladas'].max()
suma_por_año = resumen_anual.groupby('Year')['Total_Sales'].sum()
print("¿Último acumulado == suma del año?", (ultimo_acumulado_por_año.round(2) == suma_por_año.round(2)).all())
print()

# --- Validación 3: yoy_growth_categoria ---
print("=== yoy_growth_categoria ===")
print("Filas:", len(ventas_anuales_cat))
print("Combinaciones únicas esperadas:", df.groupby(['Category','Year']).ngroups)
print(f"Suma Total_Sales: {ventas_anuales_cat['Total_Sales'].sum():,.2f}")
print("¿Coincide con original?", abs(ventas_anuales_cat['Total_Sales'].sum() - total_original) < 0.01)
print("NaN esperados en YoY_Growth_Pct (solo año 2015):", ventas_anuales_cat['YoY_Growth_Pct'].isna().sum(), "de 3 esperados")
print()

# --- Validación 4: ranking_regiones ---
print("=== ranking_regiones ===")
print("Filas:", len(ranking_regiones), "(esperado: 4 regiones)")
print(f"Suma Total_Sales: {ranking_regiones['Total_Sales'].sum():,.2f}")
print("¿Coincide con original?", abs(ranking_regiones['Total_Sales'].sum() - total_original) < 0.01)
print()

# --- Validación 5: estacionalidad_categoria ---
print("=== estacionalidad_categoria ===")
print("Filas:", len(estacionalidad), "(esperado: 4 quarters x 3 categorías = 12)")
print("Combinaciones únicas esperadas:", df.groupby(['Quarter_Label','Category']).ngroups)
# Nota: esta tabla usa promedio (mean), no suma, por lo que no se compara contra total_original
print()

print("Validaciones completadas")

Suma total Sales (dataset original): 2,261,536.78

=== resumen_region_category_quarter ===
Filas: 192
Combinaciones únicas esperadas: 192
Suma Total_Sales: 2,261,536.78
¿Coincide con original? True

=== ventas_acumuladas_anual ===
Filas: 16
Combinaciones únicas esperadas: 16
Suma Total_Sales (sin acumular): 2,261,536.78
¿Coincide con original? True
¿Último acumulado == suma del año? True

=== yoy_growth_categoria ===
Filas: 12
Combinaciones únicas esperadas: 12
Suma Total_Sales: 2,261,536.78
¿Coincide con original? True
NaN esperados en YoY_Growth_Pct (solo año 2015): 3 de 3 esperados

=== ranking_regiones ===
Filas: 4 (esperado: 4 regiones)
Suma Total_Sales: 2,261,536.78
¿Coincide con original? True

=== estacionalidad_categoria ===
Filas: 12 (esperado: 4 quarters x 3 categorías = 12)
Combinaciones únicas esperadas: 12

Validaciones completadas


In [12]:
# --------------------------------------------
# 7. Exportar fuentes para Tableau
# --------------------------------------------
output_path = '../data/processed/'

df.to_csv(output_path + 'train_clean_enriched.csv', index=False)
resumen_rcq.to_csv(output_path + 'resumen_region_category_quarter.csv', index=False)
resumen_anual.to_csv(output_path + 'ventas_acumuladas_anual.csv', index=False)
ventas_anuales_cat.to_csv(output_path + 'yoy_growth_categoria.csv', index=False)
ranking_regiones.to_csv(output_path + 'ranking_regiones.csv', index=False)
estacionalidad.to_csv(output_path + 'estacionalidad_categoria.csv', index=False)

print("Archivos exportados correctamente a", output_path)

Archivos exportados correctamente a ../data/processed/
